In [36]:
!pip install neuralforecast

# Пайплайн

In [37]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
import warnings
from coreforecast.seasonal import find_season_length
#предупреждение  печати statsmodels (verbose)
warnings.filterwarnings("ignore", category=FutureWarning, module="statsmodels")
from statsmodels.stats.multitest import multipletests
from statsmodels.tsa.stattools import grangercausalitytests
from sklearn.ensemble import IsolationForest
from statsforecast.models import (SeasonalNaive, AutoARIMA, ARIMA, AutoTheta, HoltWinters, AutoETS, AutoCES, MSTL, TBATS)
from statsforecast import StatsForecast
from sklearn.preprocessing import StandardScaler
from utilsforecast.losses import mae, rmse, smape, mase
from functools import partial
from utilsforecast.evaluation import evaluate
from sklearn.linear_model import Ridge, LinearRegression, Lasso
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from mlforecast import MLForecast
from mlforecast.target_transforms import Differences
import neuralforecast
from neuralforecast import NeuralForecast
from neuralforecast.models import LSTM, NHITS, DLinear, Informer, TFT, PatchTST
from ray import tune
from neuralforecast.auto import AutoNHITS

# Обработка данных

In [38]:
df = pd.read_csv("climate_energy.csv")

#Проверка значение
print(f'Монтонность: {df.index.is_monotonic_increasing}')
print(f'Дубликаты: {df.index.has_duplicates}')
print(f'Пропуски: {df.isna().sum().sum()}')

df['date'] = pd.to_datetime(df['date'], format='mixed', dayfirst=True)
df.set_index('date', inplace=True)
df = df.asfreq('d')

#Определение сезонности
y = df.resample('W').agg({
    'energy_consumption': 'mean',
})
y = y.iloc[1:-2]
SEASON = find_season_length(y['energy_consumption'].values, max_season_length = 55)
print(f"Сезонность для энергопотребления : {SEASON}")

Монтонность: True
Дубликаты: False
Пропуски: 0
Сезонность для энергопотребления : 52


In [39]:
#Проверка значимости для экзогенных факторов
# создадим словарь для хранения компонентов
decomposed = {}
df_ = df.copy()
system_cols = ['energy_consumption']
ex_cols = ['co2_emission', 'avg_temperature','humidity', 'urban_population', 'energy_price', 'industrial_activity_index', 'renewable_share']

# функция для многосезонного декомпозиции
def multi_seasonal_decompose(series, weekly=7, yearly=365):
    result = pd.DataFrame(index=series.index)

    # годовая сезонность
    yearly_comp = seasonal_decompose(series, period=yearly, model='additive', extrapolate_trend='freq')
    result['trend_yearly'] = yearly_comp.trend
    result['season_yearly'] = yearly_comp.seasonal
    result['resid_yearly'] = yearly_comp.resid

    return result

# применяем к системным колонкам
for col in system_cols+ex_cols:
    decomposed[col] = multi_seasonal_decompose(df_[col])
    df_[col + '_trend'] = decomposed[col]['trend_yearly']
    df_[col + '_season'] = decomposed[col]['season_yearly']
    df_[col + '_resid'] = decomposed[col]['resid_yearly']  # остатки после всех сезонностей
    df_[col + '_detrend'] = df_[col + '_season']+df_[col + '_resid']

df_resid = df_[
        [f'{c}_resid' for c in system_cols] +
        [f'{c}_resid' for c in ex_cols]].dropna()


def granger_min_pvalue(df, cause, effect, max_lag=24):
    """
    Возвращает минимальный p-value по всем лагам и позицию лага
    """
    data = df[[effect, cause]].dropna()
    results = grangercausalitytests(data, maxlag=max_lag, verbose=False)
    pvals = [results[l][0]['ssr_ftest'][1] for l in results]
    return np.min(pvals), np.argmin(pvals)

# Списки причин и следствий
causes = [f'{c}_resid' for c in ex_cols]
effects = [f'{c}_resid' for c in system_cols]

# Собираем результаты
results = []

for cause in causes:
    for effect in effects:
        res = granger_min_pvalue(df_resid, cause=cause, effect=effect, max_lag=24)
        pval, lag = res
        results.append({
            'cause': cause,
            'effect': effect,
            'pval': pval,
            'lag': lag
        })

# Создаём DataFrame
granger_df = pd.DataFrame(results)

# Вытаскиваем p-значения (игнорируя NaN)
pvals = granger_df['pval'].dropna()
original_idx = granger_df.dropna().index

# Применяем FDR-коррекцию (Benjamini-Hochberg)
_, pvals_corrected, _, _ = multipletests(pvals.values, alpha=0.05, method='fdr_bh')

# Добавляем откорректированные p-значения в DataFrame
granger_df.loc[original_idx, 'pval_corrected'] = pvals_corrected

# Фильтруем значимые результаты (после коррекции)
significant = granger_df[granger_df['pval_corrected'] < 0.05]

# Выводим результаты
if not significant.empty:
    print("Значимые Granger-причинности после FDR-коррекции (p < 0.05):")
    for _, row in significant.iterrows():
        print(f"{row['cause']} → {row['effect']} | p_corrected: {row['pval_corrected']:.3e}, lag: {row['lag']}")
else:
    print("Нет значимых Granger-причинностей после FDR-коррекции.")

Значимые Granger-причинности после FDR-коррекции (p < 0.05):
avg_temperature_resid → energy_consumption_resid | p_corrected: 4.773e-02, lag: 4
humidity_resid → energy_consumption_resid | p_corrected: 2.460e-02, lag: 22
urban_population_resid → energy_consumption_resid | p_corrected: 2.460e-02, lag: 17
energy_price_resid → energy_consumption_resid | p_corrected: 2.460e-02, lag: 12


In [40]:
#Удалим неинформативные столбцы и сохраним обработанную таблицу
df = df.drop(columns = ['industrial_activity_index', 'renewable_share', 'co2_emission'])
df.to_csv('climate_EDA.csv', index_label="date")

# Анализ аномалий

In [41]:
df = pd.read_csv('climate_EDA.csv', parse_dates=["date"], index_col = "date")
ts = ['energy_consumption']
exog_cols = ['avg_temperature', 'humidity','urban_population', 'energy_price']
base_cols = ['unique_id', 'ds', 'y']

y = df.resample('W').agg({
    'energy_consumption': 'mean',
    'avg_temperature': 'mean',
    'urban_population': 'mean',
    'humidity': 'mean',
    'energy_price': 'sum',
})

y = y.iloc[1:-2]

date_index = y.index

panel_list = [
    pd.DataFrame({
        'ds': y.index,
        'y': y[t].values,
        'unique_id': t,
        # Добавляем внешние переменные из y:
        'avg_temperature': y['avg_temperature'].values,
        'urban_population': y['urban_population'].values,
        'humidity': y['humidity'].values,
        'energy_price': y['energy_price'].values
    })
    for t in ts
]

panel_df = pd.concat(panel_list, ignore_index=True)
panel_df = panel_df[base_cols + exog_cols]

FH = int(0.42 * y.shape[0])
df_test = panel_df[base_cols].groupby("unique_id").tail(FH)
df_train = panel_df[base_cols].drop(df_test.index).reset_index(drop=True)

FREQ = y.index.freq
SEASON = 52

In [42]:
levels = [99]
models = [MSTL(season_length=[SEASON], alias='MSTL')]
alias = [x.alias for x in models]

sf = StatsForecast(
    models=models,
    freq=FREQ,
)

fcst = sf.forecast(df=df_train, h=FH, level=levels, fitted=True)

insample_forecasts = sf.forecast_fitted_values().drop(columns=['y'])

res_df = panel_df.merge(pd.concat([insample_forecasts,fcst],), on=['unique_id', 'ds'], how='left')
res_df = res_df.drop(columns=['humidity', 'urban_population',	'energy_price', 'avg_temperature'],)
res_df['residual'] = res_df['y'] - res_df['MSTL']
anomalies_mstl = res_df[~res_df['y'].between(res_df['MSTL-lo-99'], res_df['MSTL-hi-99'])][['unique_id', 'ds', 'y']].copy()
anomalies_mstl['MSTL_anomaly'] = 1

In [43]:
def detect_isolation_forest_anomlies(df):
  for lag in [7, 26, 52]:
    df[f'lag_{lag}'] = df['y'].shift(lag)
    df[f'lag_{lag}'] = df['y'].shift(lag)
  scaler = StandardScaler()
  df_scaled = scaler.fit_transform(df)
  iso_forest = IsolationForest(n_estimators = 100, contamination=0.01, random_state=42)
  iso_predict = iso_forest.fit_predict(df_scaled)
  anomalies = df.index[iso_predict == -1]
  return anomalies

df_iso_en = panel_df[['ds', 'y']].copy().set_index('ds')

anomalies_if = detect_isolation_forest_anomlies(df_iso_en)

In [44]:
def detect_iqr_anomalies(df):
  Q1 = df.quantile(0.25)
  Q3 = df.quantile(0.75)
  IQR = Q3 - Q1
  lower_bound = Q1 - 1.5 * IQR
  upper_bound = Q3 + 1.5 * IQR
  df_iqr = (df < lower_bound) | (df > upper_bound)
  return df_iqr

df_en = panel_df[['ds', 'y']].copy().set_index('ds')
result_iqr = detect_iqr_anomalies(df_en)
anomalies_iqr = result_iqr[result_iqr['y']==True]

In [45]:
dates_mstl = [d.strftime('%Y-%m-%d') for d in list(anomalies_mstl['ds'])]
dates_iqr = [d.strftime('%Y-%m-%d') for d in list(anomalies_iqr.index)]
print(f'Даты аномалий вероятностного прогнозирования: {dates_mstl}')
print(f'Даты аномалий IQR: {dates_iqr}')
print(f'Даты аномалий IsolationForest: {list(anomalies_if.strftime("%Y-%m-%d"))}')

Даты аномалий вероятностного прогнозирования: ['2021-03-14', '2023-03-12', '2023-03-19', '2023-04-02', '2023-04-23', '2023-08-06', '2023-08-13', '2023-10-01', '2024-01-07', '2024-02-11', '2024-03-03', '2024-03-24', '2024-04-21', '2024-05-05']
Даты аномалий IQR: []
Даты аномалий IsolationForest: ['2022-10-02', '2023-04-23', '2023-10-01']


# Статистические методы

In [46]:
metrics = [mae, rmse, smape]
season_mase = partial(mase, seasonality=SEASON)
metrics +=[season_mase]

In [47]:
models=[
        ARIMA (order=(4,0,3), season_length=SEASON, seasonal_order=(1, 0, 0)),
        AutoARIMA(season_length=SEASON),
        SeasonalNaive(season_length=SEASON),
        #AutoTheta(season_length=SEASON, decomposition_type="multiplicative"),
        HoltWinters(season_length=SEASON, error_type="A", alias="HW_Add"),
        AutoETS(season_length=SEASON, model='ZZA', damped = False, alias='ETS'),
        AutoCES(season_length=SEASON),
        MSTL(season_length=SEASON,  ),
        TBATS(season_length = SEASON, use_boxcox = True, bc_lower_bound = 0.0, bc_upper_bound = 1, use_trend = True, use_damped_trend = True, use_arma_errors = False,),
  ]

sf_base = StatsForecast(
    models=models,
    freq=FREQ,
)
LEVELS = [75, 90, 95]

fcst_sf = sf_base.forecast(df=df_train, h=FH, level=LEVELS)
eval_sf = df_test.merge(fcst_sf, on=['unique_id', 'ds'])

metrics_base = evaluate(
    df=eval_sf,
    train_df=df_train,
    metrics=metrics,
).set_index('metric')

metrics_base

,unique_id,ARIMA,AutoARIMA,SeasonalNaive,HW_Add,ETS,CES,MSTL,TBATS
metric,,,,,,,,,
mae,energy_consumption,1608.015888,1553.884484,1553.884484,1490.834253,1404.472916,1260.755972,1237.005219,1507.221232
rmse,energy_consumption,1930.449427,2057.273367,2057.273367,1856.122255,1794.053698,1682.901981,1635.991179,1814.893991
smape,energy_consumption,0.110520,0.105761,0.105761,0.104693,0.094610,0.084417,0.083022,0.104074
mase,energy_consumption,1.099409,1.062400,1.062400,1.019292,0.960246,0.861986,0.845747,1.030496


In [49]:
date_features = ['week']
models={
        'lasso': Lasso(),
        'lin_reg': LinearRegression(),
        'ridge': Ridge(alpha=0.10),
        'rf': RandomForestRegressor(),
        'lgbm':LGBMRegressor(n_estimators=100, verbosity=-1),
}

LEVELS = [75, 90, 95]

fcst = MLForecast(
    models = models,
    freq=FREQ,
    date_features = date_features,
    target_transforms=[Differences([52])],
)
fcst.fit(df_train, )
fcst_sf = fcst.predict(h=FH, )

eval_mf = df_test[['unique_id', 'ds', 'y']].merge(
    fcst_sf,
    on=['unique_id', 'ds'],
    how='inner'
)

metrics = [mae, rmse, smape]
season_mase = partial(mase, seasonality=SEASON)
metrics +=[season_mase]
ml_metrics = evaluate(
    df=eval_mf,
    metrics=metrics,
    train_df=df_train
).set_index('metric')

ml_metrics

,unique_id,lasso,lin_reg,ridge,rf,lgbm
metric,,,,,,
mae,energy_consumption,1601.272106,1601.287394,1601.287286,2347.305022,1698.274581
rmse,energy_consumption,2128.696289,2128.722075,2128.721893,3021.169340,2253.682622
smape,energy_consumption,0.110353,0.110354,0.110354,0.165719,0.117148
mase,energy_consumption,1.094799,1.094809,1.094809,1.604866,1.161120


In [51]:
import torch
DEVICE = "gpu" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
DEVICE

'cpu'

In [64]:
test_size = SEASON
horizon = test_size//2
val_size  = test_size//2
#test_size = SEASON*3
#horizon = SEASON
log_path = ''

models = [
        LSTM(h=horizon, input_size = test_size, max_steps=50, scaler_type='standard', encoder_hidden_size=128,  decoder_hidden_size=128, default_root_dir=log_path),
        NHITS(h=horizon, input_size = test_size, max_steps=50, n_freq_downsample=[1, 1, 1], default_root_dir=log_path),
        TFT(h=horizon, input_size = test_size, max_steps=50, hidden_size=128),
        PatchTST(h=horizon, input_size = test_size, max_steps=50,),
]

nf = NeuralForecast(models=models, freq=FREQ)
nf.fit(df=df_train, val_size=val_size)
y_hat = nf.predict(df_train)

eval_dl = df_test[['unique_id', 'ds', 'y']].merge(
    y_hat,
    on=['unique_id', 'ds'],
    how='inner'
)

dl_metrics = evaluate(
    df=eval_dl,
    metrics=metrics,
    train_df=df_train
).set_index('metric')

dl_metrics

INFO:lightning_fabric.utilities.seed:Seed set to 1
INFO:lightning_fabric.utilities.seed:Seed set to 1
INFO:lightning_fabric.utilities.seed:Seed set to 1
INFO:lightning_fabric.utilities.seed:Seed set to 1
/usr/local/lib/python3.12/dist-packages/neuralforecast/common/_base_model.py:602: UserWarning: val_check_steps is greater than max_steps, setting val_check_steps to max_steps.
  warnings.warn(
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.callbacks.model_summary:
  | Name         | Type          | Params | Mode 
-------------------------------------------------------
0 | loss         | MAE           | 0      | train
1 | padder_train | ConstantPad1d | 0      | train
2 | scaler       | TemporalNorm  | 0      | train
3 | hist_encoder | LSTM          | 199 K  | train
4 | mlp_decoder  | MLP           | 16.6 K | train
--------------------------------------

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores


Predicting: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores


Predicting: |          | 0/? [00:00<?, ?it/s]

,unique_id,LSTM,NHITS,TFT,PatchTST
metric,,,,,
mae,energy_consumption,1421.615384,1575.204614,2317.342120,1407.112245
rmse,energy_consumption,1814.843928,1980.707195,2904.462762,1752.308804
smape,energy_consumption,0.092234,0.103706,0.160199,0.092696
mase,energy_consumption,0.971966,1.076976,1.584380,0.962051
